## 1. Setup and Artefact Load.
Prepare the training environment and load the unified dataset artefacts created during `01_data_eda.ipynb`.
#### Rationale:
- Establish a **reproducible** experimental context (fixed random seed and deterministic cuDNN - NVIDIA CUDA Deep Neural Network library).
- Resolve canonical project paths and verify existence of required directories (`data`, `experiments/pools/`, etc.).
- Load the **pooled table** (`source_pool_rel.csv`) and the **preprocessing configuration** (`preprocessing_config.json`) that define the 3-class problem (`Normal`, `Pneumonia`, `TB`).
- Create the output directories for checkpoints and metrics to keep training runs auditable.
- Print a concise summary confirming *GPU availability*, *ImageNet normalisation*, and *reproducibility* settings.

In [1]:
from pathlib import Path
import os, json, random, numpy as np, pandas as pd, torch

# 1. Reproducibility.
#   Use a fixed seed across all random libraries (Python, NumPy, and PyTorch)
# to make training results numerically repeatable on this workstation and future reruns.
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True # Disable non-deterministic kernels.
    torch.backends.cudnn.benchmark = False # Prevent cuDNN autotune drift.

SEED = 42
set_seed(SEED)

# 2. Resolve canonical project paths.
#   This notebook lives in notebooks/, therefore ".." points to the repository root.
PROJECT_ROOT    = Path("..").resolve()
CHEXPERT_PATH   = PROJECT_ROOT / "data" / "chexpert"
TBX11K_PATH     = PROJECT_ROOT / "data" / "tbx11k"
POOLS_DIR       = PROJECT_ROOT / "experiments" / "pools"
CHECKPOINTS_DIR = PROJECT_ROOT / "experiments" / "checkpoints"
RESULTS_DIR     = PROJECT_ROOT / "experiments" / "results"

# 3. Load persisted artefacts from 01_data_EDA.
#   The pooled CSV contains relative paths, class labels, and source tags.
# The JSON holds preprocessing constants (IMG_SIZE, ImageNet mean/std, label map, transform mode).
pool_csv = POOLS_DIR / "source_pool_rel.csv"
cfg_path = POOLS_DIR / "preprocessing_config.json"

if not pool_csv.exists() or not cfg_path.exists():
    raise FileNotFoundError("Missing pooled artefacts. Please run 01_data_eda.ipynb first "
                            "to generate experiments/pools/source_pool_rel.csv and preprocessing_config.json."
    )
source_pool_rel = pd.read_csv(pool_csv)
cfg = json.loads(cfg_path.read_text())

CLASS_TO_INDEX = {k: int(v) for k, v in cfg["class_to_index"].items()}
INDEX_TO_CLASS = {v: k for k, v in CLASS_TO_INDEX.items()}
IMG_SIZE       = int(cfg["img_size"])
IMAGENET_MEAN  = cfg["imagenet_mean"]
IMAGENET_STD   = cfg["imagenet_std"]
TRANSFORM_MODE = cfg.get("transform_mode", "center-crop")

# 5. Device and precision information.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(device.type == "cuda") # Enable mixed precision if a GPU is present.

# 6. Compact readiness log.
print("03_model_training - Environment")
print(f'- Device: {device.type.upper()} | PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
if device.type == "cuda":
    print(f'- GPU: {torch.cuda.get_device_name(0)} | cuDNN: {torch.backends.cudnn.version()}')

print("\nInputs (from 01/02):")
print(f'- Pool rows: {len(source_pool_rel):,}')
print(f'- Class map: {CLASS_TO_INDEX}')
print(f'- Image size: {IMG_SIZE} | Transform mode: {TRANSFORM_MODE}')
print(f'- ImageNet mean/std: {IMAGENET_MEAN} / {IMAGENET_STD}')

print("\nOutputs (will be written):")
print(f'- Checkpoints: {CHECKPOINTS_DIR.relative_to(PROJECT_ROOT)}')
print(f'- Metrics: {RESULTS_DIR.relative_to(PROJECT_ROOT)}')

print("\nReproducibility:")
print(f'- Seed: {SEED} | Deterministic cuDNN: {torch.backends.cudnn.deterministic}')
print(f'- Precision: {"Mixed (AMP)" if USE_AMP else "FP32"}')

03_model_training - Environment
- Device: CUDA | PyTorch: 2.10.0.dev20250930+cu128 | CUDA: 12.8
- GPU: NVIDIA GeForce RTX 5090 Laptop GPU | cuDNN: 91002

Inputs (from 01/02):
- Pool rows: 13,651
- Class map: {'Pneumonia': 0, 'TB': 1, 'Normal': 2}
- Image size: 256 | Transform mode: letterbox
- ImageNet mean/std: [0.485, 0.456, 0.406] / [0.229, 0.224, 0.225]

Outputs (will be written):
- Checkpoints: experiments/checkpoints
- Metrics: experiments/results

Reproducibility:
- Seed: 42 | Deterministic cuDNN: True
- Precision: Mixed (AMP)


In [7]:
from src.transforms import build_shared_transforms

train_transforms_shared, eval_transforms_shared = build_shared_transforms(
    img_size=IMG_SIZE,
    imagenet_mean=IMAGENET_MEAN,
    imagenet_std=IMAGENET_STD,
)